# Continuous-focusing lattice 

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize

from orbit.core.bunch import Bunch
from orbit.core.bunch import BunchTwissAnalysis
from orbit.core.spacecharge import SpaceChargeCalc2p5D
from orbit.bunch_utils import collect_bunch
from orbit.envelope import Envelope
from orbit.envelope import EnvelopeTracker
from orbit.space_charge.sc2p5d import SC2p5D_AccNode
from orbit.space_charge.sc2p5d import setSC2p5DAccNodes
from orbit.teapot import ContinuousLinearFocusingTEAPOT
from orbit.teapot import TEAPOT_Lattice
from orbit.utils.consts import mass_proton

from utils import track_env_tbt
from utils import track_bunch_tbt

plt.style.use("../style.mplstyle")

## Matched envelope calculation

In [ ]:
def calc_eq_radius(perveance: float, emittance: float, omega_0: float) -> float:
    if perveance == 0:
        return calc_eq_radius_emitt_dom(emittance, omega_0)

    radius = 1.0
    radius *= np.sqrt(0.5 * perveance) / omega_0
    radius *= np.sqrt(1.0 + np.sqrt(1.0 + (4.0 * emittance * omega_0 / perveance)**2))
    return radius

def calc_eq_radius_emitt_dom(emittance: float, omega_0: float) -> float:
    return np.sqrt(emittance / omega_0)

def calc_eq_radius_sc_dom(perveance: float, omega_0: float) -> float:
    return np.sqrt(perveance) / omega_0

def calc_depressed_freq(perveance: float, omega_0: float, radius: float, pad: float = 1e-12) -> float:
    return np.sqrt(omega_0**2 - perveance / (radius**2 + pad))

In [ ]:
class EquilibriumParameters:
    def __init__(self, length: float, omega_0: float, omega: float, emittance: float) -> None:
        self.length = length
        self.omega_0 = omega_0  # frequency
        self.sigma_0 = omega_0 / self.length  # phase advance
        self.emittance = emittance  # rms [mm mrad]
        
        self.perveance = None
        self.omega = None
        self.sigma = None
        self.radius = None
        self.set_depressed_frequency(omega)
    
    def set_depressed_frequency(self, omega: float) -> float:
        
        def loss_func(perveance: float, omega_targ: float) -> float:
            radius = calc_eq_radius(perveance, self.emittance, self.omega_0)
            omega_calc = calc_depressed_freq(perveance, self.omega_0, radius)
            return abs(omega_calc - omega_targ)
    
        result = scipy.optimize.minimize_scalar(
            loss_func, 
            bounds=(0.0, 1.0), 
            args=(omega,),
            method="bounded", 
            options=dict(xatol=1e-12)
        )
        self.perveance = result.x
        self.radius = calc_eq_radius(self.perveance, self.emittance, self.omega_0)
        self.omega = calc_depressed_freq(self.perveance, self.omega_0, self.radius)
        self.sigma = self.omega / self.length

    def set_frequency(self, omega_0: float) -> None:
        self.omega_0 = omega_0
        self.sigma_0 = omega_0 / self.length
        self.set_depressed_frequency(self.omega)

    def cov_matrix(self) -> np.ndarray:
        cov_matrix = np.zeros((4, 4))
        cov_matrix[0, 0] = 0.25 * self.radius ** 2
        cov_matrix[2, 2] = cov_matrix[0, 0]
        cov_matrix[1, 1] = self.emittance**2 / cov_matrix[0, 0]
        cov_matrix[3, 3] = cov_matrix[1, 1]
        return cov_matrix

In [ ]:
length = 0.25
sigma_0 = math.radians(80.0)
omega_0 = sigma_0 * length
omega = 0.5 * omega_0

emittance = 0.25e-06

eq_params = EquilibriumParameters(
    length=length, 
    omega_0=omega_0, 
    omega=omega, 
    emittance=emittance,
)
print("sigma =", math.degrees(eq_params.sigma))
print("sigma =", math.degrees(eq_params.sigma_0))
print("sigma / sigma_0 =", eq_params.sigma / eq_params.sigma_0)

## Track envelope

In [ ]:
kin_energy = 0.0025
bunch_length = 10.0

node = ContinuousLinearFocusingTEAPOT(
    length=eq_params.length,
    nparts=20,
    kq=(eq_params.omega_0 ** 2),
)
lattice = TEAPOT_Lattice()
lattice.addNode(node)
lattice.initialize()

cov_matrix = np.zeros((6, 6))
cov_matrix[0:4, 0:4] = eq_params.cov_matrix() 
cov_matrix[4, 4] = bunch_length**2 / 12.0

bunch = Bunch()
bunch.mass(mass_proton)
bunch.getSyncParticle().kinEnergy(kin_energy)

envelope = Envelope(bunch=bunch, cov_matrix=cov_matrix)
envelope.set_intensity(
    eq_params.perveance 
    * bunch_length
    * (envelope.beta()**2 * envelope.gamma()**3) 
    / (2.0 * envelope.classical_radius)
)

history = track_env_tbt(envelope, lattice, copy=True, turns=100)

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(history["r"])
ax.set_ylim(0.0, np.mean(history["r"]) * 1.8)
ax.set_xlabel("Period")
ax.set_ylabel("Radius")

## Track bunch

In [ ]:
size = 10_000
coords = np.random.normal(size=(size, 4))
coords = coords / np.linalg.norm(coords, axis=1)[:, None]
coords = coords / np.std(coords, axis=0)
for axis in range(4):
    coords[:, axis] *= envelope.rms(axis)

coords = np.column_stack([coords, np.zeros((size, 2))])
coords[:, 4] = 0.5 * bunch_length * np.random.uniform(-1.0, 1.0, size=size)

bunch = Bunch()
bunch.mass(mass_proton)
bunch.getSyncParticle().kinEnergy(kin_energy)
for i in range(size):
    bunch.addParticle(*coords[i])

bunch.macroSize(float(envelope.intensity) / size)

node = ContinuousLinearFocusingTEAPOT(
    length=eq_params.length,
    nparts=20,
    kq=(eq_params.omega_0 ** 2),
)
lattice = TEAPOT_Lattice()
lattice.addNode(node)
lattice.initialize()

sc_calc = SpaceChargeCalc2p5D(64, 64, 1)
sc_path_length_min = 0.001
sc_nodes = setSC2p5DAccNodes(lattice, sc_path_length_min, sc_calc)

history = track_bunch_tbt(bunch, lattice, copy=True, turns=100)

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(history["r"])
ax.set_ylim(0.0, np.mean(history["r"]) * 1.8)
ax.set_xlabel("Period")
ax.set_ylabel("Radius")